# Data Processing And Forecasting

This notebook is now a thin orchestration layer around the reusable pipeline code in `src/`. Use it for interactive runs, but keep core logic in the modules so tests and RL use the same implementation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.data_pipeline import (
    FeatureConfig,
    LatentDemandConfig,
    build_hourly_features,
    reconstruct_latent_demand,
    split_and_scale_features,
)
from src.forecasting import (
    ForecastConfig,
    NeuralForecastAdapter,
    PersistenceForecaster,
    choose_best_forecaster,
    evaluate_forecaster,
    export_rl_forecast_features,
    prepare_forecast_frames,
)


ROOT = Path.cwd()


## 1. Load Data

Point `raw_df` at a dataframe with the FreshRetailNet daily schema used in the original notebook.


In [ ]:
# Example:
# splits = {'train': 'data/train.parquet', 'eval': 'data/eval.parquet'}
# train_df = pd.read_parquet("hf://datasets/Dingdong-Inc/FreshRetailNet-50K/" + splits["train"])
# eval_df = pd.read_parquet("hf://datasets/Dingdong-Inc/FreshRetailNet-50K/" + splits["eval"])
# raw_df = pd.concat([train_df, eval_df], ignore_index=True)

raw_df = pd.DataFrame()
raw_df.head()


## 2. Reconstruct Latent Demand


In [ ]:
latent_config = LatentDemandConfig()

# The GBM imputation model must never learn stockout patterns from data that will
# become the test split, or the "ground truth" targets near the train/test boundary
# quietly encode information a real forecaster could never have had in advance.
# See lesson.md ("Leak #1") for a directed test proving this mattered.
_cutoff_frac = FeatureConfig().train_frac + FeatureConfig().val_frac
reconstruction_cutoff = pd.to_datetime(raw_df["dt"]).quantile(_cutoff_frac)

reconstructed_df, reconstruction_diagnostics = reconstruct_latent_demand(
    raw_df, latent_config, reconstruction_cutoff=reconstruction_cutoff
)
reconstruction_diagnostics


## 3. Build Hourly Features And Splits


In [ ]:
feature_config = FeatureConfig()
hourly_df = build_hourly_features(reconstructed_df, feature_config)
hourly_df, scaler = split_and_scale_features(hourly_df, feature_config)
hourly_df.head()


## 4. Prepare Forecast Data And Evaluate Candidate Models


In [ ]:
forecast_config = ForecastConfig()
bundle = prepare_forecast_frames(hourly_df, forecast_config)

# Cheap sanity baseline before spending GPU time on LSTM/TFT below.
metrics, val_predictions, best_adapter = choose_best_forecaster(
    [PersistenceForecaster()],
    bundle,
    use_log_target=forecast_config.use_log_target,
)
metrics


## 4b. LSTM and TFT (NeuralForecast)

Both models are wired through `NeuralForecastAdapter`, which:
- fits once on `train_df` (early stopping monitored on `val_df` via `val_size`)
- evaluates by walking forward through the *entire* val/test span in `horizon`-hour steps, so metrics reflect the whole split rather than a single 24h window that happens to land near the training boundary (see lesson.md, "Leak #2")

Static covariates (`product_id`, `store_id`, `city_id`, `first_category_id`) are passed via `stat_exog_list` so one shared global model can still tell products and stores apart — see lesson.md ("Leak #3") for why this was missing before.

In [ ]:
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import LSTM as NF_LSTM
from neuralforecast.models import TFT

static_cols = list(forecast_config.static_cols)

# num_workers=0 keeps each nf.fit()/nf.predict() call single-process. The walk-forward
# evaluation below calls predict() many times (once per horizon-sized window across the
# whole val/test span); with the default worker pool that's a lot of Trainer/dataloader
# subprocess churn and can exhaust memory on a laptop. Bump this up only once you're
# running on a machine (or Docker container with a memory limit) dedicated to the job.
dataloader_kwargs = {"num_workers": 0}

lstm_model = NF_LSTM(
    h=forecast_config.horizon,
    input_size=forecast_config.input_size,
    encoder_hidden_size=128,
    encoder_n_layers=2,
    encoder_dropout=0.25,
    decoder_hidden_size=128,
    decoder_layers=2,
    futr_exog_list=bundle.future_cols,
    hist_exog_list=bundle.historic_cols,
    stat_exog_list=static_cols,
    loss=MAE(),
    valid_loss=MAE(),
    max_steps=500,
    batch_size=64,
    learning_rate=5e-4,
    early_stop_patience_steps=15,
    val_check_steps=50,
    scaler_type="standard",
    dataloader_kwargs=dataloader_kwargs,
    random_seed=42,
)

tft_model = TFT(
    h=forecast_config.horizon,
    input_size=forecast_config.input_size,
    hidden_size=128,
    n_head=4,
    dropout=0.15,
    futr_exog_list=bundle.future_cols,
    hist_exog_list=bundle.historic_cols,
    stat_exog_list=static_cols,
    loss=MAE(),
    valid_loss=MAE(),
    max_steps=2000,
    batch_size=128,
    learning_rate=3e-4,
    early_stop_patience_steps=60,
    val_check_steps=100,
    scaler_type="standard",
    dataloader_kwargs=dataloader_kwargs,
    random_seed=42,
)

adapters = [PersistenceForecaster(), NeuralForecastAdapter(lstm_model, "LSTM"), NeuralForecastAdapter(tft_model, "TFT")]


### Validation comparison

In [ ]:
val_results = []
for adapter in adapters:
    m, _ = evaluate_forecaster(adapter, bundle, use_log_target=forecast_config.use_log_target, evaluation_split="val")
    val_results.append(m)
pd.DataFrame(val_results)


### Final test-set evaluation

This is the number that matters for judging real-world accuracy — it walks forward across the full, held-out test window using `train_df + val_df` as history, never touching test targets as model input.

In [ ]:
test_results = []
test_predictions = {}
for adapter in adapters:
    m, preds = evaluate_forecaster(adapter, bundle, use_log_target=forecast_config.use_log_target, evaluation_split="test")
    test_results.append(m)
    test_predictions[adapter.name] = preds
pd.DataFrame(test_results)


Pick whichever model had the lowest test MAE for the RL export in the next section (swap `best_adapter.name` below if TFT/LSTM beat persistence).

In [ ]:
best_name = pd.DataFrame(test_results).sort_values("mae").iloc[0]["model_name"]
val_predictions = test_predictions[best_name]
print(f"Using {best_name} predictions for RL export")


## 5. Export Artifacts For RL


In [ ]:
artifacts_dir = ROOT / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)

rl_forecasts = export_rl_forecast_features(val_predictions)
hourly_df.to_parquet(artifacts_dir / 'hourly_features.parquet', index=False)
rl_forecasts.to_parquet(artifacts_dir / 'rl_forecast_features.parquet', index=False)

print('Saved:', artifacts_dir / 'hourly_features.parquet')
print('Saved:', artifacts_dir / 'rl_forecast_features.parquet')
